In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import StratifiedKFold


import importlib
import FXPNET_Utils
importlib.reload(FXPNET_Utils)
from FXPNET_Utils import *


seed_everything(2025)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

BASE_DIR = Path(r"F:/GenderSimplex_Project/HCP_1200")  # <-- مسیر خودت
LABEL_CSV = BASE_DIR / "subjects_sex.csv"
SUBJ_TXT  = BASE_DIR / "Final_List_Subjects_HCP.txt"

MODEL_NAME = "FuzzyProtoNet21V3Light"
CKPT_ROOT = BASE_DIR / "Final_AllResults_V2" / "saved_models"

# خروجی‌ها
OUT_ROOT = BASE_DIR / "Final_AllResults_V2" / "Interpretability_FullPipeline_V3"
OUT_PER_CASE = OUT_ROOT / "per_case"
OUT_ATLAS    = OUT_ROOT / "atlas_summary"
OUT_PER_CASE.mkdir(parents=True, exist_ok=True)
OUT_ATLAS.mkdir(parents=True, exist_ok=True)

ATLASES = ["Brainnetome", "Gordon", "Glasser"]
RUNS = [1,2,3,4]
N_SPLITS = 5

WINDOW_SIZE = 256
STEP = 128
BATCH_SIZE = 64

# مدل باید مطابق train باشد
FEAT_DIM = 256
LATENT_DIM = 32
PROTOS_PER_CLASS = 8
LAMBDA_PROTO = 0.7
INIT_SIGMA = 1.0
TEMPERATURE = 1.0

# فایل‌های سطح
SURF_DIR = BASE_DIR / "Atlases" / "fsLR_32k_surfaces"
LH_SURF = SURF_DIR / "L.inflated.32k_fs_LR.surf.gii"
RH_SURF = SURF_DIR / "R.inflated.32k_fs_LR.surf.gii"

ATLAS_DLABEL = {
    "Brainnetome": BASE_DIR / "Atlases" / "Brainnetome" /  "fsaverage.BN_Atlas.32k_fs_LR_246regions.dlabel.nii",
    "Glasser":     BASE_DIR / "Atlases" / "Glasser"     /  "Q1-Q6_RelatedValidation210.CorticalAreas_dil_Final_Final_Areas_Group_Colors_with_Atlas_ROIs2.32k_fs_LR.dlabel.nii",
    "Gordon":      BASE_DIR / "Atlases" / "Gordon"      /  "Gordon333.32k_fs_LR_Tian_Subcortex_S1.dlabel.nii",
}


In [2]:
import importlib
import utilityFunctions
importlib.reload(utilityFunctions)
from utilityFunctions import reshapeData, prepare_data_sliding_window

import AllmodelsFuzzy
importlib.reload(AllmodelsFuzzy)
from AllmodelsFuzzy import FuzzyProtoNet21V3Light


In [ ]:
df_lab = pd.read_csv(LABEL_CSV)
df_lab["Subject"] = df_lab["Subject"].astype(str)
gender_map = {"M": 0, "F": 1, "Male": 0, "Female": 1}
df_lab["Label"] = df_lab["Sex"].map(gender_map)
label_dict = dict(zip(df_lab["Subject"], df_lab["Label"]))

with open(SUBJ_TXT, "r", encoding="utf-8") as f:
    subjects_all = [x.strip() for x in f if x.strip()]

print("subjects:", len(subjects_all))


In [4]:
def build_data_for_run(ts_dict, run_id, sids):
    ts_list = []
    for sid in sids:
        X = ts_dict[sid][run_id].astype(np.float32)
        # per-ROI zscore
        X = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + 1e-12)
        ts_list.append(X)
    T_min = min(t.shape[0] for t in ts_list)
    ts_list = [t[:T_min] for t in ts_list]
    return np.stack(ts_list, axis=0).astype(np.float32), T_min

def load_case_checkpoint(atlas_name, run_id, fold_id, n_roi):
    model_path = CKPT_ROOT / atlas_name / MODEL_NAME / f"{MODEL_NAME}_run{run_id}_fold{fold_id}.pt"
    if not model_path.exists():
        raise FileNotFoundError(model_path)

    model = FuzzyProtoNet21V3Light(
        n_roi=n_roi, n_classes=2,
        feat_dim=FEAT_DIM, latent_dim=LATENT_DIM,
        protos_per_class=PROTOS_PER_CLASS,
        dropout=0.0,
        lambda_proto=LAMBDA_PROTO,
        init_sigma=INIT_SIGMA,
        temperature=TEMPERATURE,
    ).to(device)

    state = torch.load(model_path, map_location=device)
    model.load_state_dict(state, strict=True)
    model.eval()
    return model, model_path

def load_roi_names_list(txt_path: Path):
    # فایل شما فقط یک لیست است
    names=[]
    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            s=line.strip()
            if s and not s.startswith(("#","//","%")):
                names.append(s)
    return names

ROI_NAME_FILES = {
    "Brainnetome": BASE_DIR / "Atlases" / "Brainnetome" / "annotation" / "BNA246_region_names_full.txt",
    "Glasser":     BASE_DIR / "Atlases" / "Glasser"     / "annotation" / "GlasserFreesurfer_region_names_full.txt",
    "Gordon":      BASE_DIR / "Atlases" / "Gordon"      / "annotation" / "Gordon333_Tian_Subcortex_S1_3T_region_names_full.txt",
}


In [ ]:
# دو کانفیگ: Prototype-level و Decision-level
device ='cpu'
cfg_proto = PerCaseAttrConfig(level="prototype", device=device, proto_weight_by_mu=True)
cfg_dec   = PerCaseAttrConfig(level="decision", device=device, decision_mode="logit_diff", decision_target="pred")

for ATLAS_NAME in ATLASES:
    print("\n====================", ATLAS_NAME, "====================")

    # load usable subjects for this atlas
    BASE_DIR_ATLAS = BASE_DIR / "Parcellated" / ATLAS_NAME
    valid_sids, labels_list, ts_dict = [], [], {}

    for sid in subjects_all:
        if sid not in label_dict:
            continue
        ok = True
        runs_ts = {}
        for r in RUNS:
            npy_path = BASE_DIR_ATLAS / f"{sid}_run{r}_ts.npy"
            if not npy_path.exists():
                ok = False
                break
            runs_ts[r] = np.load(npy_path)
        if not ok:
            continue
        valid_sids.append(sid)
        labels_list.append(label_dict[sid])
        ts_dict[sid] = runs_ts

    labels = np.array(labels_list, dtype=int)
    print("usable:", len(valid_sids))
    if len(valid_sids) == 0:
        continue

    # KFold split (subject-level)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=2025)
    fold_splits = list(skf.split(np.arange(len(valid_sids)), labels))

    # ROI names
    roi_names = load_roi_names_list(ROI_NAME_FILES[ATLAS_NAME])
    # later we infer N_roi from data, but roi_names should match
    # we build mapping after N_roi known.

    for RUN_ID in RUNS:
        for FOLD_ID in range(1, N_SPLITS+1):
            print(f"  -> run{RUN_ID} fold{FOLD_ID}")

            train_idx, val_idx = fold_splits[FOLD_ID-1]
            val_sids = np.array(valid_sids)[val_idx]
            val_labels = labels[val_idx]

            data_val, _ = build_data_for_run(ts_dict, RUN_ID, val_sids)
            N_subj, _, N_roi = data_val.shape

            roi_idx_to_name = {i: (roi_names[i] if i < len(roi_names) else f"ROI{i}") for i in range(N_roi)}

            X_win, y_win, subj_idx_win = prepare_data_sliding_window(
                data_val, val_labels,
                window_size=WINDOW_SIZE, step=STEP,
                return_subj_index=True
            )
            if X_win.shape[0] == 0:
                print("    no windows -> skip")
                continue

            X_win = reshapeData(X_win)  # (N_win, N_roi, W)

            loader = DataLoader(
                TensorDataset(
                    torch.from_numpy(X_win).float(),
                    torch.from_numpy(y_win).long(),
                    torch.from_numpy(subj_idx_win).long(),
                ),
                batch_size=BATCH_SIZE,
                shuffle=False
            )

            try:
                model, ckpt = load_case_checkpoint(ATLAS_NAME, RUN_ID, FOLD_ID, n_roi=N_roi)
            except Exception as e:
                print("    checkpoint fail:", e)
                continue

            # ---- Save outputs in separate trees
            out_case_proto = OUT_PER_CASE / "prototype" / ATLAS_NAME / f"run{RUN_ID}_fold{FOLD_ID}"
            out_case_dec   = OUT_PER_CASE / "decision"  / ATLAS_NAME / f"run{RUN_ID}_fold{FOLD_ID}"

            # Prototype-level
            compute_and_save_percase_attribution(
                model=model, loader=loader,
                atlas_name=ATLAS_NAME, run_id=RUN_ID, fold_id=FOLD_ID,
                roi_idx_to_name=roi_idx_to_name,
                out_dir=out_case_proto,
                protos_per_class=PROTOS_PER_CLASS,
                cfg=cfg_proto,
                save_proto_stats=True
            )

            # Decision-level
            compute_and_save_percase_attribution(
                model=model, loader=loader,
                atlas_name=ATLAS_NAME, run_id=RUN_ID, fold_id=FOLD_ID,
                roi_idx_to_name=roi_idx_to_name,
                out_dir=out_case_dec,
                protos_per_class=PROTOS_PER_CLASS,
                cfg=cfg_dec,
                save_proto_stats=False
            )

print("\n✅ Stage-9 DONE for BOTH levels.")


In [ ]:
TOPK_PER_CASE = 20   # top-k voting
Q_TOPRIGHT = 0.95
MAX_LABELS = 20
TOPN_BAR = 20

import importlib
import FXPNET_Utils
importlib.reload(FXPNET_Utils)
from FXPNET_Utils    import *

for level in ["prototype", "decision"]:
    print("\n==================== LEVEL:", level, "====================")
    for ATLAS_NAME in ATLASES:
        print("\n---", ATLAS_NAME, "---")

        # build atlas summary from per-case
        df_sum = build_atlas_summary_from_saved(
            out_per_case_root=OUT_PER_CASE,
            atlas_name=ATLAS_NAME,
            level=level,
            topk_roi_per_proto=TOPK_PER_CASE
        )

        atlas_dir = OUT_ATLAS / level / ATLAS_NAME
        atlas_dir.mkdir(parents=True, exist_ok=True)

        df_sum.to_csv(atlas_dir / "atlas_roi_vote_summary.csv", index=False)

        df_sel = select_top_right(df_sum, q=Q_TOPRIGHT, max_labels=MAX_LABELS)
        df_sel.to_csv(atlas_dir / "annotated_topright_rois.csv", index=False)

        # plots
        plot_bar_top(df_sum, atlas_dir / f"bar_vote_ratio_top{TOPN_BAR}.png", ATLAS_NAME, topn=TOPN_BAR, ycol="vote_ratio")
        plot_bar_top(df_sum, atlas_dir / f"bar_score_top{TOPN_BAR}.png",      ATLAS_NAME, topn=TOPN_BAR, ycol="score")
        plot_scatter_vote_vs_evidence(df_sum, atlas_dir / "scatter_vote_vs_evidence.png", ATLAS_NAME)
        plot_scatter_topright_with_legend(df_sum, df_sel, atlas_dir / "scatter_topright_legend.png", ATLAS_NAME)

        # surface (categorical colors + legend)
        dlabel = ATLAS_DLABEL[ATLAS_NAME]
        if (not dlabel.exists()) or (not LH_SURF.exists()) or (not RH_SURF.exists()):
            print("   surface skipped (missing files)")
            continue

        # pick ONLY 5 ROIs for surface legend clarity:
        df5 = df_sel.head(5).copy()
        roi_idxs = df5["roi_idx"].astype(int).tolist()
        roi_names = df5["roi_name"].astype(str).tolist()

        lbL, lbR = build_vertex_labels_from_dlabel(dlabel)
        n_roi = int(df_sum["roi_idx"].max()) + 1
        off = infer_label_offset(lbL, lbR, n_roi=n_roi)

        left_cat  = make_categorical_vertex_map(lbL, roi_idxs, offset=off)
        right_cat = make_categorical_vertex_map(lbR, roi_idxs, offset=off)

        out_png = atlas_dir / "surface_top20_categorical.png"
        UNDERLAY_DSCALAR = Path(SURF_DIR) / "fs_LR.32k.LR.sulc.dscalar.nii"
        plot_surface_categorical_8views(
        atlas_name=ATLAS_NAME,
        level=level,
        lh_surf=LH_SURF,
        rh_surf=RH_SURF,
        left_cat=left_cat,
        right_cat=right_cat,
        roi_names=roi_names,
        out_png=out_png,
        underlay_dscalar=UNDERLAY_DSCALAR,  # <-- sulc/curv dscalar
        darkness=0.22,
        alpha=0.90
        )
        print("   saved:", out_png)

print("\n✅ Stage-10 DONE for BOTH levels.")


In [ ]:
# ============================================================
# Stage-11 (OVERLAP-ONLY): show ONLY regions shared by >=2 atlases
# - Keeps only overlap codes: 3,5,6,7   (>=2 atlases)
# - Hides single-atlas regions: 1,2,4 and background 0
# - Uses underlay from fs_LR.32k.LR.sulc/curvature dscalar if available
# - 8 views: lateral/medial/anterior/posterior for L & R
# - Exports overlap ROI-name tables (for each overlap type)
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
from nibabel.cifti2 import cifti2_axes
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from nilearn import plotting

# ----------------------------
# reuse from previous cells if available; otherwise set them
# ----------------------------
# BASE_DIR = Path(r"F:/GenderSimplex_Project/HCP_1200")
# OUT_ROOT = BASE_DIR / "Final_AllResults_V2" / "Interpretability_FullPipeline"
# OUT_ATLAS = OUT_ROOT / "atlas_summary"
# SURF_DIR = BASE_DIR / "Atlases" / "fsLR_32k_surfaces"
# LH_SURF = SURF_DIR / "L.inflated.32k_fs_LR.surf.gii"
# RH_SURF = SURF_DIR / "R.inflated.32k_fs_LR.surf.gii"
# ATLAS_DLABEL = {...}

INTERPRET_LEVELS = ["prototype", "decision"]  # will fallback to non-level if not found
TOPN_PER_ATLAS = 10
ATLASES = ["Brainnetome", "Gordon", "Glasser"]


BASE_DIR = Path(r"F:/GenderSimplex_Project/HCP_1200")  # <-- مسیر خودت
SURF_DIR = BASE_DIR / "Atlases" / "fsLR_32k_surfaces"
LH_SURF = SURF_DIR / "L.inflated.32k_fs_LR.surf.gii"
RH_SURF = SURF_DIR / "R.inflated.32k_fs_LR.surf.gii"

# خروجی‌ها
OUT_ROOT = BASE_DIR / "Final_AllResults_V2" / "Interpretability_FullPipeline_V3"
OUT_PER_CASE = OUT_ROOT / "per_case"
OUT_ATLAS    = OUT_ROOT / "atlas_summary"

ATLAS_DLABEL = {
    "Brainnetome": BASE_DIR / "Atlases" / "Brainnetome" /  "fsaverage.BN_Atlas.32k_fs_LR_246regions.dlabel.nii",
    "Glasser":     BASE_DIR / "Atlases" / "Glasser"     /  "Q1-Q6_RelatedValidation210.CorticalAreas_dil_Final_Final_Areas_Group_Colors_with_Atlas_ROIs2.32k_fs_LR.dlabel.nii",
    "Gordon":      BASE_DIR / "Atlases" / "Gordon"      /  "Gordon333.32k_fs_LR_Tian_Subcortex_S1.dlabel.nii",
}

# Underlay dscalar (pick one you have)
UNDERLAY_DSCALAR = Path(SURF_DIR) / "fs_LR.32k.LR.sulc.dscalar.nii"
if not UNDERLAY_DSCALAR.exists():
    UNDERLAY_DSCALAR = Path(SURF_DIR) / "fs_LR.32k.LR.curvature.dscalar.nii"

OUT_STAGE11 = Path(OUT_ATLAS).parent / "stage11_overlap_only"
OUT_STAGE11.mkdir(parents=True, exist_ok=True)

OVERLAP_CODES_KEEP = {3, 5, 6, 7}  # show only overlaps (>=2 atlases)

# ----------------------------
# CIFTI helpers
# ----------------------------
def get_brainmodel_axis(img):
    ax0 = img.header.get_axis(0)
    ax1 = img.header.get_axis(1)
    if isinstance(ax0, cifti2_axes.BrainModelAxis):
        return ax0
    if isinstance(ax1, cifti2_axes.BrainModelAxis):
        return ax1
    raise RuntimeError("BrainModelAxis not found in CIFTI header.")

def load_dlabel_vertex_labels(dlabel_path: Path):
    img = nib.load(str(dlabel_path))
    data = np.asanyarray(img.dataobj)
    if data.ndim == 2:
        data = data[0]
    data = np.asarray(data).squeeze().astype(np.int32)

    bm = get_brainmodel_axis(img)
    nL = bm.nvertices.get("CIFTI_STRUCTURE_CORTEX_LEFT", None)
    nR = bm.nvertices.get("CIFTI_STRUCTURE_CORTEX_RIGHT", None)
    if nL is None or nR is None:
        raise ValueError(f"{dlabel_path.name}: missing L/R cortex structures.")

    L = np.zeros(nL, dtype=np.int32)
    R = np.zeros(nR, dtype=np.int32)
    for struct, slc, sub in bm.iter_structures():
        vals = data[slc].astype(np.int32)
        if struct == "CIFTI_STRUCTURE_CORTEX_LEFT":
            L[sub.vertex] = vals
        elif struct == "CIFTI_STRUCTURE_CORTEX_RIGHT":
            R[sub.vertex] = vals
    return L, R

def load_dscalar_underlay_lr(dscalar_path: Path):
    img = nib.load(str(dscalar_path))
    data = np.asanyarray(img.dataobj)
    if data.ndim == 2:
        data = data[0]
    data = np.asarray(data).squeeze().astype(np.float32)

    bm = get_brainmodel_axis(img)
    nL = bm.nvertices.get("CIFTI_STRUCTURE_CORTEX_LEFT", None)
    nR = bm.nvertices.get("CIFTI_STRUCTURE_CORTEX_RIGHT", None)
    if nL is None or nR is None:
        raise ValueError(f"{dscalar_path.name}: missing L/R cortex structures.")

    L = np.zeros(nL, dtype=np.float32)
    R = np.zeros(nR, dtype=np.float32)

    for struct, slc, sub in bm.iter_structures():
        vals = data[slc].astype(np.float32)
        if struct == "CIFTI_STRUCTURE_CORTEX_LEFT":
            L[sub.vertex] = vals
        elif struct == "CIFTI_STRUCTURE_CORTEX_RIGHT":
            R[sub.vertex] = vals

    # normalize for plotting
    def _norm(x):
        lo, hi = np.percentile(x, [2, 98])
        if hi <= lo:
            return x
        return np.clip((x - lo) / (hi - lo), 0, 1)

    return _norm(L), _norm(R)

def infer_offset(label_L, label_R, n_roi):
    u = np.unique(np.concatenate([label_L, label_R]))
    if (1 in u) and (n_roi in u):
        return 1
    if (0 in u) and ((n_roi - 1) in u):
        return 0
    return 1

def mask_from_roi_idx(label_vec, roi_idx_list, offset):
    m = np.zeros(label_vec.shape[0], dtype=bool)
    for ridx in roi_idx_list:
        m |= (label_vec == (int(ridx) + int(offset)))
    return m

# ----------------------------
# Read selected ROIs (ROBUST PATHS)
# ----------------------------
def read_selected_rois(level: str, atlas: str, topn: int):
    fp1 = Path(OUT_ATLAS) / level / atlas / "annotated_topright_rois.csv"
    fp2 = Path(OUT_ATLAS) / atlas / "annotated_topright_rois.csv"
    if fp1.exists():
        fp = fp1
    elif fp2.exists():
        fp = fp2
    else:
        raise FileNotFoundError(f"Missing selected ROI list.\nTried:\n  {fp1}\n  {fp2}")

    df = pd.read_csv(fp)
    if "roi_idx" not in df.columns:
        raise ValueError(f"'roi_idx' missing in {fp}")
    if "roi_name" not in df.columns:
        df["roi_name"] = df["roi_idx"].apply(lambda i: f"ROI{i}")
    return df.head(topn).copy(), fp

# ----------------------------
# Overlap coding + FILTER
# ----------------------------
def overlap_code(m_bn, m_gd, m_gl):
    return (m_bn.astype(np.int32) * 1) + (m_gd.astype(np.int32) * 2) + (m_gl.astype(np.int32) * 4)

def keep_only_overlaps(code_vec: np.ndarray, keep_codes=OVERLAP_CODES_KEEP):
    out = code_vec.copy()
    keep_mask = np.isin(out, list(keep_codes))
    out[~keep_mask] = 0
    return out

def overlap_only_cmap_and_legend():
    # 0 must be transparent
    colors = [
        (0, 0, 0, 0.0),         # 0 background
        (0.58, 0.40, 0.74, 1.0), # 3 BN+Gordon (purple)
        (0.98, 0.60, 0.11, 1.0), # 5 BN+Glasser (orange)
        (0.55, 0.34, 0.29, 1.0), # 6 Gordon+Glasser (brown)
        (0.0,  0.0,  0.0,  1.0), # 7 all (black)  <-- highest priority visual
    ]
    # We need a colormap indexed by values 0..7.
    # Create a 8-entry list with only overlap colors defined, others transparent.
    full = [(0,0,0,0.0)] * 8
    full[0] = (0,0,0,0.0)
    full[3] = colors[1]
    full[5] = colors[2]
    full[6] = colors[3]
    full[7] = colors[4]
    cmap = ListedColormap(full)

    legend = [
        (3, "Brainnetome ∩ Gordon"),
        (5, "Brainnetome ∩ Glasser"),
        (6, "Gordon ∩ Glasser"),
        (7, "Brainnetome ∩ Gordon ∩ Glasser (all)"),
    ]
    patches = [Patch(facecolor=full[k], edgecolor="black", label=lab) for k, lab in legend]
    return cmap, patches

def plot_overlap_only_8views(title, L_code, R_code, L_bg, R_bg, out_png):
    cmap, patches = overlap_only_cmap_and_legend()
    fig = plt.figure(figsize=(20, 10))
    views = ["lateral", "medial", "anterior"]

    for i, view in enumerate(views):
        ax = fig.add_subplot(2, 3, i+1, projection="3d")
        plotting.plot_surf_roi(
            surf_mesh=str(LH_SURF), roi_map=L_code, bg_map=L_bg,
            cmap=cmap, hemi="left", view=view,
            colorbar=False, axes=ax,
            title=f"Left | {view}",title_font_size= 20,
            darkness=0.45, alpha=0.98
        )
        ax = fig.add_subplot(2, 3, i+4, projection="3d")
        plotting.plot_surf_roi(
            surf_mesh=str(RH_SURF), roi_map=R_code, bg_map=R_bg,
            cmap=cmap, hemi="right", view=view,
            colorbar=False, axes=ax,
            title=f"Right | {view}",title_font_size = 20,
            darkness=0.45, alpha=0.98
        )

    fig.suptitle(title, fontsize=18, y=0.98)
    fig.legend(handles=patches, loc="center right", bbox_to_anchor=(1.02, 0.5),
               frameon=True, ncol=1, fontsize=14, title="Overlap only (≥2 atlases)")
    plt.tight_layout(rect=[0, 0, 0.86, 0.95])
    fig.savefig(out_png, dpi=450, bbox_inches="tight")
    plt.close(fig)

# ----------------------------
# Tables: list ROI names that participate in each overlap code
# ----------------------------
def rois_participating_in_overlap(label_L, label_R, offset, df_sel, code_L, code_R, needed_codes):
    roi_idx = df_sel["roi_idx"].astype(int).to_list()
    roi_name = df_sel["roi_name"].astype(str).to_list()
    out = {c: [] for c in needed_codes}
    for ridx, rname in zip(roi_idx, roi_name):
        lab = ridx + offset
        mL = (label_L == lab)
        mR = (label_R == lab)
        for c in needed_codes:
            if np.any(code_L[mL] == c) or np.any(code_R[mR] == c):
                out[c].append(rname)
    return out

def make_overlap_name_table(level, df_bn, df_gd, df_gl,
                            lb_bn_L, lb_bn_R, lb_gd_L, lb_gd_R, lb_gl_L, lb_gl_R,
                            off_bn, off_gd, off_gl,
                            code_L, code_R):
    needed = [3,5,6,7]
    label_of_code = {
        3: "Brainnetome ∩ Gordon",
        5: "Brainnetome ∩ Glasser",
        6: "Gordon ∩ Glasser",
        7: "Brainnetome ∩ Gordon ∩ Glasser",
    }
    bn_map = rois_participating_in_overlap(lb_bn_L, lb_bn_R, off_bn, df_bn, code_L, code_R, needed)
    gd_map = rois_participating_in_overlap(lb_gd_L, lb_gd_R, off_gd, df_gd, code_L, code_R, needed)
    gl_map = rois_participating_in_overlap(lb_gl_L, lb_gl_R, off_gl, df_gl, code_L, code_R, needed)

    rows = []
    for c in needed:
        rows.append({
            "level": level,
            "overlap_code": c,
            "overlap_type": label_of_code[c],
            "Brainnetome_ROIs_in_overlap": ", ".join(bn_map[c]) if bn_map[c] else "-",
            "Gordon_ROIs_in_overlap":      ", ".join(gd_map[c]) if gd_map[c] else "-",
            "Glasser_ROIs_in_overlap":     ", ".join(gl_map[c]) if gl_map[c] else "-",
        })
    return pd.DataFrame(rows)

# ============================================================
# RUN
# ============================================================

# surface existence
for p in [LH_SURF, RH_SURF]:
    if not Path(p).exists():
        raise FileNotFoundError(f"Missing surface: {p}")

# underlay load
if UNDERLAY_DSCALAR.exists():
    L_bg, R_bg = load_dscalar_underlay_lr(UNDERLAY_DSCALAR)
    print("✅ Underlay:", UNDERLAY_DSCALAR)
else:
    print("⚠ Underlay dscalar not found. Running WITHOUT underlay.")
    L_bg, R_bg = None, None

all_tables = []

for LEVEL in INTERPRET_LEVELS:
    print(f"\n==============================")
    print(f"Stage-11 OVERLAP-ONLY | LEVEL={LEVEL}")
    print(f"==============================")

    # read selected ROI lists
    df_bn, fp_bn = read_selected_rois(LEVEL, "Brainnetome", TOPN_PER_ATLAS)
    df_gd, fp_gd = read_selected_rois(LEVEL, "Gordon", TOPN_PER_ATLAS)
    df_gl, fp_gl = read_selected_rois(LEVEL, "Glasser", TOPN_PER_ATLAS)

    print("Using ROI lists:")
    print("  BN:", fp_bn)
    print("  GD:", fp_gd)
    print("  GL:", fp_gl)

    roi_bn = df_bn["roi_idx"].astype(int).to_list()
    roi_gd = df_gd["roi_idx"].astype(int).to_list()
    roi_gl = df_gl["roi_idx"].astype(int).to_list()

    # dlabels -> vertex labels
    lb_bn_L, lb_bn_R = load_dlabel_vertex_labels(ATLAS_DLABEL["Brainnetome"])
    lb_gd_L, lb_gd_R = load_dlabel_vertex_labels(ATLAS_DLABEL["Gordon"])
    lb_gl_L, lb_gl_R = load_dlabel_vertex_labels(ATLAS_DLABEL["Glasser"])

    # offsets
    off_bn = infer_offset(lb_bn_L, lb_bn_R, n_roi=int(df_bn["roi_idx"].max()) + 1)
    off_gd = infer_offset(lb_gd_L, lb_gd_R, n_roi=int(df_gd["roi_idx"].max()) + 1)
    off_gl = infer_offset(lb_gl_L, lb_gl_R, n_roi=int(df_gl["roi_idx"].max()) + 1)

    # masks per atlas
    m_bn_L = mask_from_roi_idx(lb_bn_L, roi_bn, off_bn)
    m_bn_R = mask_from_roi_idx(lb_bn_R, roi_bn, off_bn)
    m_gd_L = mask_from_roi_idx(lb_gd_L, roi_gd, off_gd)
    m_gd_R = mask_from_roi_idx(lb_gd_R, roi_gd, off_gd)
    m_gl_L = mask_from_roi_idx(lb_gl_L, roi_gl, off_gl)
    m_gl_R = mask_from_roi_idx(lb_gl_R, roi_gl, off_gl)

    # vertex counts must match
    if not (m_bn_L.shape == m_gd_L.shape == m_gl_L.shape and m_bn_R.shape == m_gd_R.shape == m_gl_R.shape):
        raise ValueError("Vertex counts do not match across atlases. Ensure all dlables are fsLR 32k.")

    # overlap codes (0..7)
    L_code = overlap_code(m_bn_L, m_gd_L, m_gl_L)
    R_code = overlap_code(m_bn_R, m_gd_R, m_gl_R)

    # KEEP ONLY overlaps >=2 atlases
    L_code = keep_only_overlaps(L_code, keep_codes=OVERLAP_CODES_KEEP)
    R_code = keep_only_overlaps(R_code, keep_codes=OVERLAP_CODES_KEEP)

    # plot (8 views)
    out_png = OUT_STAGE11 / f"OVERLAP_ONLY_{LEVEL}_Top{TOPN_PER_ATLAS}_8views.png"
    plot_overlap_only_8views(
        title=f"Overlap-only regions (≥2 atlases) | Top-{TOPN_PER_ATLAS} per atlas | {LEVEL}-level",
        L_code=L_code, R_code=R_code,
        L_bg=L_bg, R_bg=R_bg,
        out_png=out_png
    )
    print("✅ Saved:", out_png)

    # tables: ROI names in each overlap type
    df_tab = make_overlap_name_table(
        level=LEVEL,
        df_bn=df_bn, df_gd=df_gd, df_gl=df_gl,
        lb_bn_L=lb_bn_L, lb_bn_R=lb_bn_R,
        lb_gd_L=lb_gd_L, lb_gd_R=lb_gd_R,
        lb_gl_L=lb_gl_L, lb_gl_R=lb_gl_R,
        off_bn=off_bn, off_gd=off_gd, off_gl=off_gl,
        code_L=L_code, code_R=R_code
    )
    all_tables.append(df_tab)
    out_csv = OUT_STAGE11 / f"OVERLAP_ONLY_ROI_NAMES_{LEVEL}_Top{TOPN_PER_ATLAS}.csv"
    df_tab.to_csv(out_csv, index=False)
    print("✅ Saved:", out_csv)

# combined table
df_all = pd.concat(all_tables, ignore_index=True)
out_all = OUT_STAGE11 / f"OVERLAP_ONLY_ROI_NAMES_BOTHLEVELS_Top{TOPN_PER_ATLAS}.csv"
df_all.to_csv(out_all, index=False)

print("\n✅ Saved combined:", out_all)
print("✅ Stage-11 OVERLAP-ONLY DONE ->", OUT_STAGE11)


In [ ]:
# ============================================================
# Stage-11b: Precise overlap-to-ROI naming (per atlas)
# - Fixes offset inference (Gordon often offset=0)
# - For each overlap code (3,5,6,7), finds which ROI labels
#   truly contain overlap vertices, and reports:
#   roi_idx, roi_name, overlap_vertices, roi_size_vertices, overlap_percent
# - Saves 3 CSVs per level (BN/Gordon/Glasser) + a combined CSV
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

OVERLAP_CODES = [3,5,6,7]
CODE_NAME = {
    3: "BN∩Gordon",
    5: "BN∩Glasser",
    6: "Gordon∩Glasser",
    7: "BN∩Gordon∩Glasser",
}

def infer_offset_fixed(label_L, label_R, n_roi):
    u = np.unique(np.concatenate([label_L, label_R]))
    if (1 in u) and (n_roi in u):
        return 1
    if (0 in u) and ((n_roi - 1) in u):
        return 0   # ✅ IMPORTANT FIX
    return 1

def build_roi_overlap_table_for_atlas(
    atlas: str,
    df_sel: pd.DataFrame,        # has roi_idx, roi_name (topN list)
    lab_L: np.ndarray,
    lab_R: np.ndarray,
    offset: int,
    L_code: np.ndarray,
    R_code: np.ndarray,
    overlap_codes = OVERLAP_CODES
):
    """
    For a given atlas, returns a table listing ROI entries that actually contain overlap vertices.
    We compute overlap stats per ROI label.
    """
    rows = []
    roi_map = dict(zip(df_sel["roi_idx"].astype(int), df_sel["roi_name"].astype(str)))

    for roi_idx, roi_name in roi_map.items():
        lab_val = int(roi_idx) + int(offset)

        roi_mask_L = (lab_L == lab_val)
        roi_mask_R = (lab_R == lab_val)

        roi_size = int(roi_mask_L.sum() + roi_mask_R.sum())
        if roi_size == 0:
            continue

        for c in overlap_codes:
            ov_L = int(np.sum(roi_mask_L & (L_code == c)))
            ov_R = int(np.sum(roi_mask_R & (R_code == c)))
            ov = ov_L + ov_R
            if ov == 0:
                continue

            rows.append({
                "atlas": atlas,
                "overlap_code": c,
                "overlap_type": CODE_NAME[c],
                "roi_idx": int(roi_idx),
                "label_value_in_dlabel": lab_val,
                "roi_name": roi_name,
                "overlap_vertices": ov,
                "roi_total_vertices": roi_size,
                "overlap_percent": 100.0 * ov / max(roi_size, 1),
            })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df
    df = df.sort_values(["overlap_code","overlap_vertices","overlap_percent"], ascending=[True,False,False]).reset_index(drop=True)
    return df

# where to save
OUT_STAGE11B = Path(OUT_STAGE11) / "overlap_name_debug"
OUT_STAGE11B.mkdir(parents=True, exist_ok=True)

all_combined = []

for LEVEL in INTERPRET_LEVELS:
    print(f"\n==============================")
    print(f"Stage-11b | LEVEL={LEVEL}")
    print(f"==============================")

    # read selected
    df_bn, _ = read_selected_rois(LEVEL, "Brainnetome", TOPN_PER_ATLAS)
    df_gd, _ = read_selected_rois(LEVEL, "Gordon", TOPN_PER_ATLAS)
    df_gl, _ = read_selected_rois(LEVEL, "Glasser", TOPN_PER_ATLAS)

    # load labels
    lb_bn_L, lb_bn_R = load_dlabel_vertex_labels(ATLAS_DLABEL["Brainnetome"])
    lb_gd_L, lb_gd_R = load_dlabel_vertex_labels(ATLAS_DLABEL["Gordon"])
    lb_gl_L, lb_gl_R = load_dlabel_vertex_labels(ATLAS_DLABEL["Glasser"])

    # FIXED offsets
    off_bn = infer_offset_fixed(lb_bn_L, lb_bn_R, n_roi=int(df_bn["roi_idx"].max()) + 1)
    off_gd = infer_offset_fixed(lb_gd_L, lb_gd_R, n_roi=int(df_gd["roi_idx"].max()) + 1)
    off_gl = infer_offset_fixed(lb_gl_L, lb_gl_R, n_roi=int(df_gl["roi_idx"].max()) + 1)

    print("Offsets (fixed):", "BN=", off_bn, "Gordon=", off_gd, "Glasser=", off_gl)

    # build masks for overlap coding (same as Stage-11)
    roi_bn = df_bn["roi_idx"].astype(int).to_list()
    roi_gd = df_gd["roi_idx"].astype(int).to_list()
    roi_gl = df_gl["roi_idx"].astype(int).to_list()

    m_bn_L = mask_from_roi_idx(lb_bn_L, roi_bn, off_bn)
    m_bn_R = mask_from_roi_idx(lb_bn_R, roi_bn, off_bn)
    m_gd_L = mask_from_roi_idx(lb_gd_L, roi_gd, off_gd)
    m_gd_R = mask_from_roi_idx(lb_gd_R, roi_gd, off_gd)
    m_gl_L = mask_from_roi_idx(lb_gl_L, roi_gl, off_gl)
    m_gl_R = mask_from_roi_idx(lb_gl_R, roi_gl, off_gl)

    L_code = overlap_code(m_bn_L, m_gd_L, m_gl_L)
    R_code = overlap_code(m_bn_R, m_gd_R, m_gl_R)
    L_code = keep_only_overlaps(L_code, keep_codes=OVERLAP_CODES_KEEP)
    R_code = keep_only_overlaps(R_code, keep_codes=OVERLAP_CODES_KEEP)

    # Now build precise tables
    t_bn = build_roi_overlap_table_for_atlas("Brainnetome", df_bn, lb_bn_L, lb_bn_R, off_bn, L_code, R_code)
    t_gd = build_roi_overlap_table_for_atlas("Gordon",     df_gd, lb_gd_L, lb_gd_R, off_gd, L_code, R_code)
    t_gl = build_roi_overlap_table_for_atlas("Glasser",    df_gl, lb_gl_L, lb_gl_R, off_gl, L_code, R_code)

    # Save
    out_dir = OUT_STAGE11B / LEVEL
    out_dir.mkdir(parents=True, exist_ok=True)

    t_bn.to_csv(out_dir / f"OverlapROI_ExactNames_Brainnetome_Top{TOPN_PER_ATLAS}.csv", index=False)
    t_gd.to_csv(out_dir / f"OverlapROI_ExactNames_Gordon_Top{TOPN_PER_ATLAS}.csv", index=False)
    t_gl.to_csv(out_dir / f"OverlapROI_ExactNames_Glasser_Top{TOPN_PER_ATLAS}.csv", index=False)

    # combined
    comb = pd.concat([t_bn, t_gd, t_gl], ignore_index=True) if (len(t_bn)+len(t_gd)+len(t_gl))>0 else pd.DataFrame()
    if len(comb) > 0:
        comb.insert(0, "level", LEVEL)
        comb.to_csv(out_dir / f"OverlapROI_ExactNames_ALLATLASES_Top{TOPN_PER_ATLAS}.csv", index=False)
        all_combined.append(comb)

    print("✅ Saved tables to:", out_dir)
    display(t_bn.head(10))
    display(t_gd.head(10))
    display(t_gl.head(10))

if len(all_combined) > 0:
    df_all = pd.concat(all_combined, ignore_index=True)
    df_all.to_csv(OUT_STAGE11B / f"OverlapROI_ExactNames_BOTHLEVELS_Top{TOPN_PER_ATLAS}.csv", index=False)
    print("\n✅ Saved combined BOTH-level table:", OUT_STAGE11B / f"OverlapROI_ExactNames_BOTHLEVELS_Top{TOPN_PER_ATLAS}.csv")
else:
    print("\n⚠ No overlaps found in the selected Top-N sets.")


In [ ]:
# ============================================================
# TOP-20 ROIs on fsLR-32k surface (4 views) with sharp sulcal underlay
# - Works for BOTH levels: "prototype" and "decision"
# - Reads selected ROIs from:
#     OUT_ATLAS/<level>/<atlas>/annotated_topright_rois.csv
#   or fallback:
#     OUT_ATLAS/<atlas>/annotated_topright_rois.csv
# - Plots 4 views per hemisphere: lateral/medial/anterior/posterior
# - Underlay uses your fs_LR.32k.LR.sulc.dscalar.nii (or curvature) for folds
# - Saves one figure per (level, atlas)
#
# Put this in a NEW Jupyter cell AFTER you have:
#   OUT_ATLAS, SURF_DIR, LH_SURF, RH_SURF, ATLAS_DLABEL
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
from nibabel.cifti2 import cifti2_axes
import matplotlib.pyplot as plt
from nilearn import plotting
from matplotlib.colors import ListedColormap

# ----------------------------
# USER SETTINGS
# ----------------------------
LEVELS = ["decision", "prototype"]      # which interpret levels to plot
ATLASES = ["Brainnetome", "Gordon", "Glasser"]
TOPN_PER_ATLAS = 10                    # <-- you asked for 20

# Underlay (folds)
UNDERLAY_SULC = Path(SURF_DIR) / "fs_LR.32k.LR.sulc.dscalar.nii"
UNDERLAY_CURV = Path(SURF_DIR) / "fs_LR.32k.LR.curvature.dscalar.nii"
UNDERLAY_DSCALAR = UNDERLAY_SULC if UNDERLAY_SULC.exists() else UNDERLAY_CURV

# Output directory
OUT_STAGE_SURF = Path(OUT_ATLAS).parent / "stage_topROIs_surface"
OUT_STAGE_SURF.mkdir(parents=True, exist_ok=True)

# ----------------------------
# CIFTI helpers
# ----------------------------
def get_brainmodel_axis(img):
    ax0 = img.header.get_axis(0)
    ax1 = img.header.get_axis(1)
    if isinstance(ax0, cifti2_axes.BrainModelAxis):
        return ax0
    if isinstance(ax1, cifti2_axes.BrainModelAxis):
        return ax1
    raise RuntimeError("BrainModelAxis not found in CIFTI header.")

def load_dlabel_vertex_labels(dlabel_path: Path):
    """Return per-vertex label vectors for L/R cortex."""
    img = nib.load(str(dlabel_path))
    data = np.asanyarray(img.dataobj)
    if data.ndim == 2:
        data = data[0]
    data = np.asarray(data).squeeze().astype(np.int32)

    bm = get_brainmodel_axis(img)
    nL = bm.nvertices.get("CIFTI_STRUCTURE_CORTEX_LEFT", None)
    nR = bm.nvertices.get("CIFTI_STRUCTURE_CORTEX_RIGHT", None)
    if nL is None or nR is None:
        raise ValueError(f"{dlabel_path.name}: missing L/R cortex structures.")

    L = np.zeros(nL, dtype=np.int32)
    R = np.zeros(nR, dtype=np.int32)
    for struct, slc, sub in bm.iter_structures():
        vals = data[slc].astype(np.int32)
        if struct == "CIFTI_STRUCTURE_CORTEX_LEFT":
            L[sub.vertex] = vals
        elif struct == "CIFTI_STRUCTURE_CORTEX_RIGHT":
            R[sub.vertex] = vals
    return L, R

def load_dscalar_underlay_lr(dscalar_path: Path):
    """
    Returns L_bg, R_bg in [0,1] after robust normalization.
    Also flips/contrasts (for sulc) to make folds pop more.
    """
    img = nib.load(str(dscalar_path))
    data = np.asanyarray(img.dataobj)
    if data.ndim == 2:
        data = data[0]
    data = np.asarray(data).squeeze().astype(np.float32)

    bm = get_brainmodel_axis(img)
    nL = bm.nvertices.get("CIFTI_STRUCTURE_CORTEX_LEFT", None)
    nR = bm.nvertices.get("CIFTI_STRUCTURE_CORTEX_RIGHT", None)
    if nL is None or nR is None:
        raise ValueError(f"{dscalar_path.name}: missing L/R cortex structures.")

    L = np.zeros(nL, dtype=np.float32)
    R = np.zeros(nR, dtype=np.float32)
    for struct, slc, sub in bm.iter_structures():
        vals = data[slc].astype(np.float32)
        if struct == "CIFTI_STRUCTURE_CORTEX_LEFT":
            L[sub.vertex] = vals
        elif struct == "CIFTI_STRUCTURE_CORTEX_RIGHT":
            R[sub.vertex] = vals

    # robust normalization + contrast boost
    def _norm_contrast(x):
        lo, hi = np.percentile(x, [1, 99])
        if hi <= lo:
            return x
        x = (x - lo) / (hi - lo)
        x = np.clip(x, 0, 1)
        # gamma for sharper folds
        gamma = 0.65
        x = np.power(x, gamma)
        return x

    return _norm_contrast(L), _norm_contrast(R)

def infer_offset_fixed(label_L, label_R, n_roi):
    """
    If labels look like 1..n_roi => offset=1
    If labels look like 0..n_roi-1 => offset=0
    """
    u = np.unique(np.concatenate([label_L, label_R]))
    if (1 in u) and (n_roi in u):
        return 1
    if (0 in u) and ((n_roi - 1) in u):
        return 0
    return 1

def read_selected_rois(level: str, atlas: str, topn: int):
    """
    Tries:
      1) OUT_ATLAS/<level>/<atlas>/annotated_topright_rois.csv
      2) OUT_ATLAS/<atlas>/annotated_topright_rois.csv
    """
    fp1 = Path(OUT_ATLAS) / level / atlas / "annotated_topright_rois.csv"
    fp2 = Path(OUT_ATLAS) / atlas / "annotated_topright_rois.csv"
    if fp1.exists():
        fp = fp1
    elif fp2.exists():
        fp = fp2
    else:
        raise FileNotFoundError(f"Missing selected ROI list.\nTried:\n  {fp1}\n  {fp2}")

    df = pd.read_csv(fp)
    if "roi_idx" not in df.columns:
        raise ValueError(f"'roi_idx' missing in {fp}")
    if "roi_name" not in df.columns:
        df["roi_name"] = df["roi_idx"].apply(lambda i: f"ROI{i}")
    # keep topN by current order (your Stage-10 already sorted by score)
    return df.head(topn).copy(), fp

# ----------------------------
# Build a categorical ROI map: each ROI gets its own ID (1..K)
# (so they get different colors on the surface)
# ----------------------------
def build_categorical_roi_map(label_vec, roi_idx_list, offset):
    """
    Returns int array size n_vertices:
      0 background
      j+1 for ROI j in roi_idx_list order
    """
    out = np.zeros(label_vec.shape[0], dtype=np.int32)
    for j, ridx in enumerate(roi_idx_list):
        lab = int(ridx) + int(offset)
        out[label_vec == lab] = int(j + 1)
    return out

def make_topk_colormap(k):
    """
    Background transparent, then k colors from tab20 repeated if needed.
    """
    import matplotlib.cm as cm
    base = cm.get_cmap("tab20")
    colors = [(0,0,0,0.0)]
    for i in range(k):
        colors.append(base(i % base.N))
    return ListedColormap(colors)

# ----------------------------
# Plot 8 panels (4 views x 2 hemispheres)
# ----------------------------
def plot_top_rois_8views(
    atlas_name: str,
    level: str,
    roi_names: list,
    roi_map_L: np.ndarray,
    roi_map_R: np.ndarray,
    L_bg: np.ndarray,
    R_bg: np.ndarray,
    out_png: Path,
    darkness: float = 0.25,    # smaller => folds more visible
    alpha: float = 0.88
):
    views = ["lateral", "medial", "anterior", "posterior"]
    k = len(roi_names)
    cmap = make_topk_colormap(k)

    fig = plt.figure(figsize=(22, 10))

    for i, view in enumerate(views):
        ax = fig.add_subplot(2, 4, i+1, projection="3d")
        plotting.plot_surf_roi(
            surf_mesh=str(LH_SURF), roi_map=roi_map_L, bg_map=L_bg,
            cmap=cmap, hemi="left", view=view,
            colorbar=False, axes=ax,
            title=f"Left | {view}",
            darkness=darkness, alpha=alpha
        )

        ax = fig.add_subplot(2, 4, i+5, projection="3d")
        plotting.plot_surf_roi(
            surf_mesh=str(RH_SURF), roi_map=roi_map_R, bg_map=R_bg,
            cmap=cmap, hemi="right", view=view,
            colorbar=False, axes=ax,
            title=f"Right | {view}",
            darkness=darkness, alpha=alpha
        )

    # legend (ROI -> color)
    from matplotlib.patches import Patch
    patches = []
    for j, nm in enumerate(roi_names):
        patches.append(Patch(facecolor=cmap(j+1), edgecolor="black", label=f"{j+1:02d}. {nm}"))

    fig.suptitle(f"{atlas_name} | {level}-level | Top-{k} ROIs (categorical colors)", fontsize=16, y=0.99)
    fig.legend(handles=patches, loc="center right", bbox_to_anchor=(1.02, 0.5),
               frameon=True, ncol=1, fontsize=9, title="ROI colors")
    plt.tight_layout(rect=[0, 0, 0.84, 0.96])
    fig.savefig(out_png, dpi=450, bbox_inches="tight")
    plt.close(fig)

# ----------------------------
# RUN
# ----------------------------
# sanity checks
for p in [LH_SURF, RH_SURF]:
    if not Path(p).exists():
        raise FileNotFoundError(f"Missing surface: {p}")

if not UNDERLAY_DSCALAR.exists():
    raise FileNotFoundError(f"Underlay dscalar not found. Tried:\n  {UNDERLAY_SULC}\n  {UNDERLAY_CURV}")

L_bg, R_bg = load_dscalar_underlay_lr(UNDERLAY_DSCALAR)
print("✅ Underlay used:", UNDERLAY_DSCALAR)

for level in LEVELS:
    for atlas in ATLASES:
        print(f"\n==================== {atlas} | {level} ====================")

        df_sel, fp = read_selected_rois(level, atlas, TOPN_PER_ATLAS)
        print("Selected ROI list:", fp)

        dlabel_path = Path(ATLAS_DLABEL[atlas])
        if not dlabel_path.exists():
            raise FileNotFoundError(f"Missing dlabel for {atlas}: {dlabel_path}")

        lab_L, lab_R = load_dlabel_vertex_labels(dlabel_path)

        n_roi = int(df_sel["roi_idx"].max()) + 1
        offset = infer_offset_fixed(lab_L, lab_R, n_roi=n_roi)
        print(f"Offset({atlas}) = {offset}")

        roi_idx_list = df_sel["roi_idx"].astype(int).tolist()
        roi_names = df_sel["roi_name"].astype(str).tolist()

        roi_map_L = build_categorical_roi_map(lab_L, roi_idx_list, offset)
        roi_map_R = build_categorical_roi_map(lab_R, roi_idx_list, offset)

        out_png = OUT_STAGE_SURF / f"Top{TOPN_PER_ATLAS}_{atlas}_{level}_8views.png"
        plot_top_rois_8views(
            atlas_name=atlas,
            level=level,
            roi_names=roi_names,
            roi_map_L=roi_map_L,
            roi_map_R=roi_map_R,
            L_bg=L_bg,
            R_bg=R_bg,
            out_png=out_png,
            darkness=0.22,   # <-- makes folds clearer
            alpha=0.90
        )
        print("✅ Saved:", out_png)

print("\n✅ DONE: Top-ROIs surface figures saved under:", OUT_STAGE_SURF)
